In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GATConv
from sklearn.metrics import r2_score
import numpy as np
import random

def set_seed(seed=42):
    """Set all random generators used by the training workflow."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class SimpleGAT(nn.Module):
    """GAT regressor with optional edge and graph-level features."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2, gat_heads=4):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims must not be empty")
        if any(h_dim % gat_heads != 0 for h_dim in hidden_dims):
            raise ValueError("Each hidden dimension must be divisible by gat_heads")

        self.gat_heads = gat_heads
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None
        self.global_mlp = None
        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        self.convs = nn.ModuleList()
        self.conv_norms = nn.ModuleList()
        in_dim = node_dim
        for h_dim in hidden_dims:
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=h_dim // gat_heads,
                heads=gat_heads,
                concat=True,
                dropout=dropout,
                edge_dim=edge_dim if edge_dim else None,
                add_self_loops=True
            ))
            self.conv_norms.append(nn.LayerNorm(h_dim))
            in_dim = h_dim

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        edge_attr = getattr(data, 'edge_attr', None)
        if self.edge_norm is not None and edge_attr is not None:
            edge_attr = self.edge_norm(edge_attr)

        u = getattr(data, 'u', None)
        if self.global_norm is not None and u is not None:
            u = self.global_norm(u)

        for conv, conv_norm in zip(self.convs, self.conv_norms):
            x = conv(x, data.edge_index, edge_attr=edge_attr)
            x = conv_norm(x)
            x = F.elu(x)
            x = self.dropout(x)

        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze(-1)
        return (out, h) if return_feat else out

def create_data_loader(graph_data, batch_size=32, shuffle=True):
    """Convert stored graph dictionaries to PyG Data objects."""
    data_list = []
    for graph in graph_data:
        data_list.append(Data(
            x=graph['x'],
            edge_index=graph['edge_index'],
            edge_attr=graph.get('edge_attr', None),
            u=graph.get('u', None),
            y=graph['y']
        ))
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

def train_model(
    train_data_dir: str,
    val_data_dir: str,
    save_path: str,
    epochs=1000,
    batch_size=32,
    lr=1e-4,
    min_lr=1e-6,
    hidden_dims=[64, 64],
    dropout=0.2,
    gat_heads=4,
    lr_patience=10,
    es_patience=100,
):
    """Train one baseline model and save the checkpoint with the best validation R²."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_graph_data = torch.load(os.path.join(train_data_dir, "graph_data.pt"))
    val_graph_data = torch.load(os.path.join(val_data_dir, "graph_data.pt"))

    train_y = torch.stack([g['y'] for g in train_graph_data]).view(-1)
    y_mean, y_std = train_y.mean().item(), train_y.std().item() + 1e-8

    for g in train_graph_data:
        g['y'] = (g['y'] - y_mean) / y_std
    for g in val_graph_data:
        g['y'] = (g['y'] - y_mean) / y_std

    train_loader = create_data_loader(train_graph_data, batch_size, True)
    val_loader = create_data_loader(val_graph_data, batch_size, False)

    sample = train_graph_data[0]
    node_dim = sample['x'].size(1)
    edge_dim = sample.get('edge_attr').size(1) if sample.get('edge_attr') is not None else 0
    global_dim = sample.get('u').size(1) if sample.get('u') is not None else 0

    model = SimpleGAT(node_dim, edge_dim, global_dim, hidden_dims, dropout, gat_heads).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=lr_patience, min_lr=min_lr
    )

    best_r2, no_improve = -float('inf'), 0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            pred, _ = model(batch, return_feat=True)
            loss = F.mse_loss(pred, batch.y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        ys_train, preds_train = [], []
        with torch.no_grad():
            for batch in train_loader:
                batch = batch.to(device)
                out = model(batch)
                ys_train.append(batch.y.view(-1).cpu().numpy())
                preds_train.append(out.cpu().numpy())
        train_r2 = r2_score(np.concatenate(ys_train), np.concatenate(preds_train))

        ys_val, preds_val = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                ys_val.append(batch.y.view(-1).cpu().numpy())
                preds_val.append(out.cpu().numpy())
        val_r2 = r2_score(np.concatenate(ys_val), np.concatenate(preds_val))

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch} | Loss: {epoch_loss/len(train_loader):.4f} | "
              f"Train R²: {train_r2:.4f} | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")
        scheduler.step(epoch_loss)

        if val_r2 > best_r2:
            best_r2 = val_r2
            no_improve = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'y_mean': y_mean,
                'y_std': y_std,
                'node_dim': node_dim,
                'edge_dim': edge_dim,
                'global_dim': global_dim,
                'hidden_dims': hidden_dims,
                'dropout': dropout,
                'gat_heads': gat_heads,
                'model_type': 'gat'
            }, save_path)
            print(f"Best checkpoint saved | validation R2={best_r2:.4f}")
        else:
            no_improve += 1
            if no_improve >= es_patience:
                print(f"Early stopping: validation R2 did not improve for {es_patience} epochs.")
                break

    print(f"Training complete, best Val R²={best_r2:.4f}, model saved at {save_path}")
    return best_r2, save_path


if __name__ == "__main__":
    # Repeated runs used for the reported robustness analysis.
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]

    # Set these three paths before running.
    train_data_dir = None
    val_data_dir = None
    save_dir = None
    if not all([train_data_dir, val_data_dir, save_dir]):
        raise ValueError("Set train_data_dir, val_data_dir, and save_dir before running.")

    os.makedirs(save_dir, exist_ok=True)
    for seed in seeds:
        set_seed(seed)
        save_path = os.path.join(save_dir, f"gat_baseline_seed{seed}.pt")
        print(f"Training GAT baseline | seed={seed}")
        train_model(
            train_data_dir=train_data_dir,
            val_data_dir=val_data_dir,
            save_path=save_path,
            epochs=5000,
            batch_size=64,
            lr=1e-3,
            min_lr=1e-4,
            hidden_dims=[128, 128],
            dropout=0.1,
            gat_heads=4,
            lr_patience=20,
            es_patience=100,
        )
